In [ ]:
!pip3 install toml
import warnings
warnings.filterwarnings("ignore")
import toml
import numpy as np
import joblib
import psycopg2

# Leer secretos
secrets = toml.load("secrets.toml")["postgres"]

USER = secrets["USER"]
PASSWORD = secrets["PASSWORD"]
HOST = secrets["HOST"]
PORT = secrets["PORT"]
DBNAME = secrets["DBNAME"]

# 2️⃣ Cargar modelo y scaler
model = joblib.load("components/residuos_model.pkl")
with open("components/residuos_model_info.pkl", "rb") as f:
    model_info = joblib.load(f)

feature_names = model_info["columnas"]

#Generar registros
np.random.seed(42)

data = []

for _ in range(10):

    record = [
        np.random.randint(1000, 100000),
        np.random.randint(500, 90000),
        np.random.randint(100, 20000),
        np.random.randint(2018, 2025)
    ]

    data.append(record)
# Guardar en Supabase
try:

    connection = psycopg2.connect(
        user=USER,
        password=PASSWORD,
        host=HOST,
        port=PORT,
        dbname=DBNAME
    )

    cursor = connection.cursor()

    for record in data:

        prediction = float(
            model.predict([record])[0]
        )

        columns = ", ".join(feature_names)

        placeholders = ", ".join(
            ["%s"] * (len(record) + 1)
        )

        sql = f'''
        INSERT INTO pc_ml_residuos
        ({columns}, prediction)
        VALUES ({placeholders})
        '''

        cursor.execute(
            sql,
            record + [prediction]
        )

    connection.commit()

    cursor.close()
    connection.close()

    print(
        "✅ Registros guardados en Supabase"
    )

except Exception as e:

    print(
        f"❌ Error: {e}"
    )
